# Dataset integrity with cryptographic hashes

**Module 03 — Cybersecurity Fundamentals for AI Systems**

A hash tells you whether bytes changed. It does **not** tell you who produced them — an attacker who
replaces a dataset can also replace its hash. This notebook builds a dataset manifest, detects tampering,
and then shows exactly where the hash-only approach fails and a keyed MAC succeeds.

## Learning objectives

1. Compute and verify SHA-256 digests over a dataset and a multi-file dataset directory.
2. Detect silent tampering, including a single changed label.
3. Explain the difference between integrity and authenticity, and demonstrate it.
4. Build a tamper-evident hash chain over ordered shards.
5. Apply this to a training pipeline's ingestion gate.

## Background

A cryptographic hash function maps arbitrary bytes to a fixed-length digest with three properties:
preimage resistance (you cannot invert it), second-preimage resistance (you cannot find another input
with the same digest), and collision resistance. Module 03 uses the first two to detect modification and
needs collision resistance for the chain in Step 4.

Authenticity is a different property. To prove *who* produced the bytes you need a secret: an HMAC with
a shared key, or a signature with a private key. The demonstration in Step 3 is the one students most
often get wrong in practice.

## Dependencies and installation

```bash
python -m pip install "pandas>=2.2"
```

Only the standard library is needed (`hashlib`, `hmac`, `json`, `pathlib`); pandas is used to write and
read the CSV. No network access, CPU only, under a second of runtime.

In [1]:
import hashlib
import hmac
import json
import tempfile
from pathlib import Path

import pandas as pd

SEED = 3
rng_state = 3

workdir = Path(tempfile.mkdtemp(prefix="sas-m03-"))
print("temporary workspace:", workdir.name)

temporary workspace: sas-m03-ko3z2n72


**Expected output** — a path under your system temporary directory. Nothing is written outside it, and
the directory is disposable; the notebook does not depend on files from a previous run.

### Step 1 — Create a small labelled dataset

Twelve referral records with a triage label. Small enough to inspect by eye, which matters when you are
learning to detect a one-row change.

In [2]:
records = pd.DataFrame(
    {
        "record_id": [f"R{i:03d}" for i in range(1, 13)],
        "age": [34, 51, 62, 45, 29, 71, 58, 40, 66, 23, 49, 55],
        "triage": ["green", "amber", "red", "amber", "green",
                   "red", "amber", "green", "red", "green", "amber", "amber"],
    }
)
dataset_path = workdir / "referrals.csv"
records.to_csv(dataset_path, index=False)
records.head()

,record_id,age,triage
0,R001,34,green
1,R002,51,amber
2,R003,62,red
3,R004,45,amber
4,R005,29,green


**Expected output** — the first five rows. `triage` has three values; a poisoned dataset would need to
change very few of them to matter.

### Step 2 — Hash the bytes, then verify

Hash the *file bytes*, not the parsed table: parsing is lenient, and two different files can parse to the
same frame.

In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8192), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_file(path: Path, expected: str) -> bool:
    return sha256_file(path) == expected


manifest = {"file": dataset_path.name, "sha256": sha256_file(dataset_path)}
(workdir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("verify:", verify_file(dataset_path, manifest["sha256"]))

{
  "file": "referrals.csv",
  "sha256": "e69bfda3fac42ffe2b22055d94d394f346ab8c60741b45c58374209b001ce127"
}
verify: True


**Expected output** — a 64-character hex digest and `verify: True`. The digest is a function of every
byte including newlines and column order, so it is reproducible across machines but changes on any edit.

### Step 3 — Tamper with one label and watch the digest change

One character changes. The digest does not "partially" match — it is completely different, which is the
property that makes silent modification detectable.

In [4]:
tampered = records.copy()
tampered.loc[tampered.record_id == "R007", "triage"] = "green"   # amber -> green
tampered_path = workdir / "referrals_tampered.csv"
tampered.to_csv(tampered_path, index=False)

original_digest = sha256_file(dataset_path)
tampered_digest = sha256_file(tampered_path)
print("original :", original_digest)
print("tampered :", tampered_digest)
print("match    :", original_digest == tampered_digest)
print("rows differing:", int((records.triage != tampered.triage).sum()))

original : e69bfda3fac42ffe2b22055d94d394f346ab8c60741b45c58374209b001ce127
tampered : 08f2245a9ca90e47750f85687698f3e4f8cc4982cd52aac68d34836aaab7c3dd
match    : False
rows differing: 1


**Expected output** — two unrelated digests and `match: False`, with exactly one differing row. This is
the check that belongs in an ingestion job: fail the pipeline, do not "fix" the file.

### Step 4 — Where hash-only verification fails

An attacker who can write the dataset can usually write the manifest too. Below, the attacker tampers and
recomputes the digest, and the naive check passes. This is not a bug in SHA-256; it is the wrong tool for
the question "did *our* vendor produce this?".

In [5]:
attacker_manifest = {"file": tampered_path.name, "sha256": sha256_file(tampered_path)}
print("naive check against attacker-supplied manifest:",
      verify_file(tampered_path, attacker_manifest["sha256"]))

# Keyed integrity: the key never travels with the data.
KEY = b"course-demo-key-not-a-real-secret"   # in production: a secret manager, rotated


def mac_file(path: Path, key: bytes) -> str:
    return hmac.new(key, path.read_bytes(), hashlib.sha256).hexdigest()


good_mac = mac_file(dataset_path, KEY)
tampered_mac = mac_file(tampered_path, KEY)
print("MAC matches for the tampered file:", hmac.compare_digest(good_mac, tampered_mac))
print("MAC matches for the original file:", hmac.compare_digest(good_mac, mac_file(dataset_path, KEY)))

naive check against attacker-supplied manifest: True
MAC matches for the tampered file: False
MAC matches for the original file: True


**Expected output** — `naive check ... True` (the attack succeeds) and `MAC matches ... False`. That pair
of lines is the whole lesson: hashes detect accidental change, MACs and signatures detect *untrusted*
change, and only one of those is a security control.

### Step 5 — A tamper-evident chain over shards

Large datasets arrive in shards. Hashing each shard independently lets an attacker delete a whole shard
without detection. Chaining each shard's digest into the next one — the idea behind Merkle trees and
transparency logs — makes removals and reorderings visible.

In [6]:
shard_dir = workdir / "shards"
shard_dir.mkdir()
for i in range(4):
    records.iloc[i * 3 : (i + 1) * 3].to_csv(shard_dir / f"part-{i}.csv", index=False)


def chain_digests(paths):
    prev = "0" * 64
    chain = []
    for p in paths:
        h = hashlib.sha256((prev + sha256_file(p)).encode()).hexdigest()
        chain.append((p.name, h))
        prev = h
    return chain


paths = sorted(shard_dir.glob("part-*.csv"))
chain = chain_digests(paths)
root = chain[-1][1]
print("root:", root)
print("chain length:", len(chain))

# Attacker deletes part-2 entirely; the root changes.
remaining = [p for p in paths if p.name != "part-2.csv"]
root_after = chain_digests(remaining)[-1][1]
print("root after deleting one shard:", root_after)
print("detected:", root != root_after)

root: 9c6084a436d3a04de42b8f1e1d210d6eae090dbe2ba8fda4f85dfe207b35edad
chain length: 4
root after deleting one shard: 82721496aedca4bf51d5d4ec4171f2950363a0177f93eb7dc775d7dd9c5569a7
detected: True


**Expected output** — a root digest, then a different root once a shard is removed, with
`detected: True`. If you only checked per-shard digests, `part-0`, `part-1` and `part-3` would all still
verify while a quarter of the data was gone.

## Exercises

1. **Reorder the chain.** Swap shards 1 and 2 before chaining and show that the root changes. Why does
   ordering matter for a dataset?
2. **Add a vendor signature.** Sketch (in code or prose) how you would replace the shared key with a
   public-key signature so that consumers need no secret.
3. **Write the ingestion gate.** Produce a function `accept(path, manifest, mac_key) -> bool` that fails
   closed and logs the reason. Test it with the tampered file.
4. **Measure the cost.** Time `sha256_file` on a 50 MB file (generate random bytes) and extrapolate to a
   10 GB dataset. Is verification affordable on every ingestion?

## Reflection questions

1. Why is "the checksum matched" a weak statement when the checksum came from the same server?
2. A dataset passes its MAC check. Which of the three CIA properties does that establish, and which two
   does it leave untouched?
3. Where in the lifecycle from Module 02 must verification happen so that a tampered file cannot reach
   training?

## References

1. NIST, *FIPS 180-4*, Secure Hash Standard.
2. NIST, *FIPS 198-1*, The Keyed-Hash Message Authentication Code (HMAC).
3. RFC 2104, *HMAC: Keyed-Hashing for Message Authentication*.
4. SLSA specification — provenance and artefact verification.